# Reasoning agent

**Dishant Kaushik (M24MA2003)**

Knowledge base + query -> internal representation -> classical reasoning -> True / False.

For the evaluation the knowledge base and the query are given directly as logic, so the main input is a set of formulas. The parser accepts the usual notations (FOLIO style `∀x (P(x) → Q(x))`, ASCII `forall x (p(x) -> q(x))`, and Prolog style `q(X) :- p(X).`). How the formulas are used:

1. Horn theories (facts and *for all x: conjunction -> literal* rules, negation allowed) go to forward chaining with negation as failure, which is the closed world reading RuleTaker uses
2. everything else (or, xor, iff, exists, ...) is turned into clauses and proved by resolution refutation with best-first clause selection, open world
3. counterfactual questions over a causal model (CounterBench) use the structural equations: abduction, intervention, forward propagation

English input still works through two parsers (a pattern parser for simple theories and a spaCy based one for FOLIO style text), and every prediction still comes from the classical reasoning code. Set the environment variable `TEST_FILE` to the test file before running. With `TEST_FILE` set, the experiments and traces on the public data (sections 7 and 8) are skipped and nothing is downloaded; set `RUN_EXPERIMENTS=1` to run them as well.

In [1]:
import os
import re
import json
import time
import heapq
import itertools
from collections import defaultdict, Counter

import pandas as pd
from IPython.display import display

DATA_DIR = os.environ.get("DATA_DIR", "data")
TEST_FILE = os.environ.get("TEST_FILE", "")
RUN_EXPERIMENTS = os.environ.get("RUN_EXPERIMENTS", "0" if TEST_FILE else "1") == "1"
NEED_PUBLIC_DATA = RUN_EXPERIMENTS or not TEST_FILE


## 1. Logic: formulas, parser, clause form

Formulas are nested tuples, e.g. `('all', '?x', ('imp', ('atom', 'young', ('?x',)), ('atom', 'smart', ('?x',))))`. Free single letter variables (x, y, z, ...) are read as universally quantified, in Prolog clauses capitalised arguments are variables. Predicate names are compared in lower case. The words of a CamelCase name (`FemaleTennisPlayer` -> female, tennis, player) are kept on the side for the word subset axioms (tested in section 7).

In [2]:
TOKEN_RE = re.compile(r"\s*(<->|<=>|->|=>|&&|\|\||!=|/\\|\\/|\\\+|∀|∃|¬|∧|∨|→|↔|⟷|⊕|⇒|⇔|≠|=|\(|\)|,|~|!|&|\||\^|-|:|;|\.|[\w'][\w'.\-]*)")

SYMBOLS = {
    "<->": "↔", "<=>": "↔", "⟷": "↔", "⇔": "↔", "iff": "↔",
    "->": "→", "=>": "→", "⇒": "→", "implies": "→",
    "&&": "∧", "&": "∧", "/\\": "∧", "and": "∧",
    "||": "∨", "|": "∨", "\\/": "∨", "or": "∨",
    "~": "¬", "!": "¬", "-": "¬", "\\+": "¬", "not": "¬",
    "^": "⊕", "xor": "⊕",
    "!=": "≠",
}
QUANTS = {"forall": "∀", "all": "∀", "exists": "∃", "exist": "∃"}
OPERATORS = {"∀", "∃", "¬", "∧", "∨", "→", "↔", "⊕", "≠", "=", "(", ")", ",", ":", ";", "."}


def tokenize(s):
    s = s.replace("—", "-").replace("’", "'").strip()
    raw, pos = [], 0
    while pos < len(s):
        m = TOKEN_RE.match(s, pos)
        if not m:
            pos += 1
            continue
        tok = m.group(1)
        if len(tok) > 1 and tok.endswith(".") and tok[0] not in ".":
            tok = tok.rstrip(".")
        raw.append(tok)
        pos = m.end()
    toks = []
    for i, t in enumerate(raw):
        low = t.lower()
        nxt = raw[i + 1] if i + 1 < len(raw) else None
        if low in QUANTS and nxt is not None and nxt not in OPERATORS and low not in SYMBOLS:
            toks.append(QUANTS[low])
        elif t in SYMBOLS or (low in ("and", "or", "not", "iff", "implies", "xor") and nxt != "("):
            toks.append(SYMBOLS.get(t, SYMBOLS.get(low)))
        elif t in (":", ";", "."):
            continue
        else:
            toks.append(t)
    return toks


PRED_WORDS = {}


def pred_key(name):
    key = name.lower()
    if key not in PRED_WORDS:
        split = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", name).lower()
        PRED_WORDS[key] = frozenset(w for w in split.split("_") if w)
    return key


def pred_words(key):
    return PRED_WORDS.get(key) or frozenset(w for w in key.split("_") if w)


class FOLParser:
    def __init__(self, text, upper_vars=False):
        self.toks = tokenize(text)
        self.i = 0
        self.bound = []
        self.free = []
        self.upper_vars = upper_vars

    def peek(self):
        return self.toks[self.i] if self.i < len(self.toks) else None

    def take(self, expected=None):
        t = self.peek()
        if expected is not None and t != expected:
            raise SyntaxError(f"expected {expected} got {t}")
        self.i += 1
        return t

    def parse(self):
        f = self.iff()
        if self.peek() is not None:
            raise SyntaxError("trailing tokens: " + " ".join(self.toks[self.i:]))
        return f

    def iff(self):
        f = self.imp()
        while self.peek() == "↔":
            self.take()
            f = ("iff", f, self.imp())
        return f

    def imp(self):
        f = self.xor()
        if self.peek() == "→":
            self.take()
            return ("imp", f, self.imp())
        return f

    def xor(self):
        f = self.disj()
        while self.peek() == "⊕":
            self.take()
            f = ("xor", f, self.disj())
        return f

    def disj(self):
        f = self.conj()
        while self.peek() == "∨":
            self.take()
            f = ("or", f, self.conj())
        return f

    def conj(self):
        f = self.unary()
        while self.peek() == "∧":
            self.take()
            f = ("and", f, self.unary())
        return f

    def unary(self):
        t = self.peek()
        if t == "¬":
            self.take()
            return ("not", self.unary())
        if t in ("∀", "∃"):
            self.take()
            v = self.take()
            if v is None or v in OPERATORS:
                raise SyntaxError("quantifier without a variable")
            self.bound.append(v)
            if self.peek() in ("∀", "∃", "¬", "("):
                body = self.unary()
                if self.peek() in ("→", "∧", "∨", "⊕", "↔") and v in self.toks[self.i:]:
                    body = self.continue_binary(body)
            else:
                body = self.iff()
            self.bound.pop()
            return ("all" if t == "∀" else "ex", "?" + v, body)
        if t == "(":
            self.take()
            f = self.iff()
            self.take(")")
            return f
        return self.atom()

    def continue_binary(self, left):
        op = self.take()
        right = self.iff()
        name = {"→": "imp", "∧": "and", "∨": "or", "⊕": "xor", "↔": "iff"}[op]
        return (name, left, right)

    def term(self):
        name = self.take()
        if self.peek() == "(":
            self.take()
            args = [self.term()]
            while self.peek() == ",":
                self.take()
                args.append(self.term())
            self.take(")")
            return (name,) + tuple(args)
        if name in self.bound:
            return "?" + name
        if self.is_free_var(name):
            if name not in self.free:
                self.free.append(name)
            return "?" + name
        return name.lower()

    def is_free_var(self, name):
        if self.upper_vars:
            return name[0].isupper() or name[0] == "_"
        return re.fullmatch(r"[u-z][0-9]*", name) is not None

    def atom(self):
        name = self.take()
        if name is None:
            raise SyntaxError("unexpected end")
        if self.peek() == "(":
            self.take()
            args = [self.term()] if self.peek() != ")" else []
            while self.peek() == ",":
                self.take()
                args.append(self.term())
            self.take(")")
            left = None
            f = ("atom", pred_key(name), tuple(args))
        else:
            left = "?" + name if name in self.bound else name.lower()
            f = ("atom", pred_key(name), ())
        if self.peek() in ("=", "≠"):
            op = self.take()
            right = self.term()
            if left is None:
                left = (f[1],) + f[2]
            f = ("atom", "=", (left, right))
            if op == "≠":
                f = ("not", f)
        return f


def balance(text):
    out, depth = [], 0
    for ch in text:
        if ch == "(":
            depth += 1
        elif ch == ")":
            if depth == 0:
                continue
            depth -= 1
        out.append(ch)
    return "".join(out) + ")" * depth


def close(f, free):
    for v in reversed(free):
        f = ("all", "?" + v, f)
    return f


def repair(text):
    text = re.sub(r"([∀∃])\s*\(", r"\1x (", text)
    text = re.sub(r"\)\s+v\s+\(", ") ∨ (", text)
    return text


def parse_one(text, upper_vars=False):
    for attempt in (text, balance(text), balance(repair(text))):
        p = FOLParser(attempt, upper_vars)
        try:
            return close(p.parse(), p.free)
        except SyntaxError as e:
            rest = p.toks[p.i:]
            if str(e).startswith("trailing tokens") and rest and rest[0] not in OPERATORS:
                p2 = FOLParser(attempt, upper_vars)
                p2.toks = p.toks[:p.i]
                try:
                    return close(p2.parse(), p2.free)
                except SyntaxError:
                    pass
            last = e
    raise last


def parse_fol(text):
    t = text.strip()
    t = re.sub(r"^\?-\s*", "", t)
    if ":-" in t:
        head, body = t.split(":-", 1)
        parts = split_args(body.strip().rstrip("."), ",")
        conj = " ∧ ".join("(" + b + ")" for b in parts)
        return parse_one(f"({conj}) → ({head.strip()})", upper_vars=True)
    return parse_one(t)


def split_args(text, sep):
    out, depth, cur = [], 0, ""
    for ch in text:
        if ch == "(":
            depth += 1
        elif ch == ")":
            depth -= 1
        if ch == sep and depth == 0:
            out.append(cur)
            cur = ""
        else:
            cur += ch
    if cur.strip():
        out.append(cur)
    return [c.strip() for c in out if c.strip()]


def split_formulas(text):
    if isinstance(text, (list, tuple)):
        return [str(x) for x in text if str(x).strip()]
    lines = [l.strip() for l in str(text).split("\n") if l.strip()]
    out = []
    for line in lines:
        line = re.sub(r"^(?:[-*•]\s+|\(?\d+[.)]\s+|(?:[A-Za-z]+\s*\d*|[A-Za-z]+\s+\d+)\s*:\s+)", "", line).strip()
        if not line:
            continue
        depth, cur = 0, ""
        for i, ch in enumerate(line):
            if ch == "(":
                depth += 1
            elif ch == ")":
                depth -= 1
            nxt = line[i + 1] if i + 1 < len(line) else " "
            if depth == 0 and (ch == ";" or (ch == "." and nxt.isspace())):
                if cur.strip():
                    out.append(cur.strip())
                cur = ""
            else:
                cur += ch
        if cur.strip():
            out.append(cur.strip().rstrip("."))
    return out


In [3]:
_fresh = itertools.count()


def elim(f):
    op = f[0]
    if op == "atom":
        return f
    if op == "not":
        return ("not", elim(f[1]))
    if op in ("all", "ex"):
        return (op, f[1], elim(f[2]))
    a, b = elim(f[1]), elim(f[2])
    if op == "imp":
        return ("or", ("not", a), b)
    if op == "iff":
        return ("and", ("or", ("not", a), b), ("or", a, ("not", b)))
    if op == "xor":
        return ("and", ("or", a, b), ("or", ("not", a), ("not", b)))
    return (op, a, b)


def nnf(f, neg=False):
    op = f[0]
    if op == "atom":
        return ("not", f) if neg else f
    if op == "not":
        return nnf(f[1], not neg)
    if op in ("and", "or"):
        new = op if not neg else ("or" if op == "and" else "and")
        return (new, nnf(f[1], neg), nnf(f[2], neg))
    if op in ("all", "ex"):
        new = op if not neg else ("ex" if op == "all" else "all")
        return (new, f[1], nnf(f[2], neg))
    raise ValueError(op)


def sub_term(t, theta):
    if isinstance(t, str):
        if t.startswith("?") and t in theta:
            return sub_term(theta[t], theta)
        return t
    return (t[0],) + tuple(sub_term(a, theta) for a in t[1:])


def rename(f, theta):
    op = f[0]
    if op == "atom":
        return ("atom", f[1], tuple(sub_term(t, theta) for t in f[2]))
    if op == "not":
        return ("not", rename(f[1], theta))
    if op in ("and", "or"):
        return (op, rename(f[1], theta), rename(f[2], theta))
    v = f[1]
    nv = f"?v{next(_fresh)}"
    return (op, nv, rename(f[2], dict(theta, **{v: nv})))


def skolem(f, univ):
    op = f[0]
    if op == "atom" or op == "not":
        return f
    if op in ("and", "or"):
        return (op, skolem(f[1], univ), skolem(f[2], univ))
    if op == "all":
        return skolem(f[2], univ + [f[1]])
    name = f"sk{next(_fresh)}"
    term = name if not univ else (name,) + tuple(univ)
    return skolem(rename_var(f[2], f[1], term), univ)


def rename_var(f, v, term):
    op = f[0]
    if op == "atom":
        return ("atom", f[1], tuple(sub_term(t, {v: term}) for t in f[2]))
    if op == "not":
        return ("not", rename_var(f[1], v, term))
    if op in ("and", "or"):
        return (op, rename_var(f[1], v, term), rename_var(f[2], v, term))
    if f[1] == v:
        return f
    return (op, f[1], rename_var(f[2], v, term))


def to_clauses(f, limit=400):

    def cnf(g):
        op = g[0]
        if op == "atom":
            return [frozenset([(True, g[1], g[2])])]
        if op == "not":
            a = g[1]
            return [frozenset([(False, a[1], a[2])])]
        if op == "and":
            return cnf(g[1]) + cnf(g[2])
        left, right = cnf(g[1]), cnf(g[2])
        if len(left) * len(right) > limit:
            raise OverflowError("cnf too big")
        return [a | b for a in left for b in right]

    g = skolem(rename(nnf(elim(f)), {}), [])
    out = []
    for c in cnf(g):
        if any((not s, p, a) in c for s, p, a in c):
            continue
        out.append(c)
    return out


### Unification and resolution

Refutation: add the negated goal and look for the empty clause. Set of support strategy, and the clause picked next is the one with the lowest cost (number of literals + term depth), which makes it a uniform cost search over derived clauses. Limits: 1 s and 3000 clauses per proof.

In [4]:
def is_var(t):
    return isinstance(t, str) and t.startswith("?")


def walk(t, theta):
    while is_var(t) and t in theta:
        t = theta[t]
    return t


def occurs(v, t, theta):
    t = walk(t, theta)
    if t == v:
        return True
    if isinstance(t, tuple):
        return any(occurs(v, a, theta) for a in t[1:])
    return False


def unify(x, y, theta):
    if theta is None:
        return None
    x, y = walk(x, theta), walk(y, theta)
    if x == y:
        return theta
    if is_var(x):
        return None if occurs(x, y, theta) else dict(theta, **{x: y})
    if is_var(y):
        return None if occurs(y, x, theta) else dict(theta, **{y: x})
    if isinstance(x, tuple) and isinstance(y, tuple) and len(x) == len(y) and x[0] == y[0]:
        for a, b in zip(x[1:], y[1:]):
            theta = unify(a, b, theta)
            if theta is None:
                return None
        return theta
    return None


def unify_args(a1, a2, theta):
    if len(a1) != len(a2):
        return None
    for x, y in zip(a1, a2):
        theta = unify(x, y, theta)
        if theta is None:
            return None
    return theta


def resolve_term(t, theta):
    t = walk(t, theta)
    if isinstance(t, tuple):
        return (t[0],) + tuple(resolve_term(a, theta) for a in t[1:])
    return t


def subst_clause(c, theta):
    return frozenset((s, p, tuple(resolve_term(a, theta) for a in args)) for s, p, args in c)


def standardize(c):
    vs = {}
    out = []
    for s, p, args in c:
        out.append((s, p, tuple(_std_term(a, vs) for a in args)))
    return frozenset(out)


def _std_term(t, vs):
    if is_var(t):
        if t not in vs:
            vs[t] = f"?r{next(_fresh)}"
        return vs[t]
    if isinstance(t, tuple):
        return (t[0],) + tuple(_std_term(a, vs) for a in t[1:])
    return t


def term_depth(t):
    if isinstance(t, tuple):
        return 1 + max(term_depth(a) for a in t[1:])
    return 0


def clause_key(c):
    names = {}
    out = []
    for s, p, args in sorted(c, key=repr):
        out.append((s, p, tuple(_canon(a, names) for a in args)))
    return tuple(sorted(out, key=repr))


def _canon(t, names):
    if is_var(t):
        if t not in names:
            names[t] = f"?{len(names)}"
        return names[t]
    if isinstance(t, tuple):
        return (t[0],) + tuple(_canon(a, names) for a in t[1:])
    return t


In [5]:
class Prover:

    def __init__(self, max_clauses=500, max_seconds=2.0, max_depth=3):
        self.max_clauses = max_clauses
        self.max_seconds = max_seconds
        self.max_depth = max_depth
        self.steps = 0

    def cost(self, c):
        return len(c) + sum(term_depth(a) for _, _, args in c for a in args)

    def refute(self, kb_clauses, sos_clauses):
        start = time.time()
        usable = []
        seen = set()
        heap = []
        counter = itertools.count()
        parents = {}
        for c in kb_clauses:
            k = clause_key(c)
            if k not in seen:
                seen.add(k)
                usable.append(c)
        for c in sos_clauses:
            k = clause_key(c)
            if k not in seen:
                seen.add(k)
                heapq.heappush(heap, (self.cost(c), next(counter), c))
                parents[k] = ("goal",)
        self.steps = 0
        given_count = len(seen)
        while heap:
            if time.time() - start > self.max_seconds or len(seen) - given_count > self.max_clauses:
                return False, None
            _, _, given = heapq.heappop(heap)
            self.steps += 1
            given = standardize(given)
            for other in usable + [given]:
                for res in self.resolvents(given, other):
                    if not res:
                        parents["empty"] = (given, other)
                        self.proof_parents = parents
                        return True, parents
                    if any(term_depth(a) > self.max_depth for _, _, args in res for a in args):
                        continue
                    if any((not s, p, a) in res for s, p, a in res):
                        continue
                    k = clause_key(res)
                    if k in seen:
                        continue
                    seen.add(k)
                    parents[k] = (given, other)
                    heapq.heappush(heap, (self.cost(res), next(counter), res))
            usable.append(given)
        return False, None

    def resolvents(self, c1, c2):
        out = []
        for s1, p1, a1 in c1:
            for s2, p2, a2 in c2:
                if s1 == s2 or p1 != p2 or len(a1) != len(a2):
                    continue
                theta = unify_args(a1, a2, {})
                if theta is None:
                    continue
                rest = (c1 - {(s1, p1, a1)}) | (c2 - {(s2, p2, a2)})
                out.append(subst_clause(rest, theta))
        return out


def equality_axioms(clauses):
    uses_eq = any(p == "=" for c in clauses for _, p, _ in c)
    if not uses_eq:
        return []
    consts = set()
    for c in clauses:
        for _, _, args in c:
            for a in args:
                if isinstance(a, str) and not is_var(a) and not a.startswith("sk"):
                    consts.add(a)
    out = [frozenset([(True, "=", ("?e", "?e"))])]
    for a, b in itertools.combinations(sorted(consts), 2):
        out.append(frozenset([(False, "=", (a, b))]))
    out.append(frozenset([(False, "=", ("?e1", "?e2")), (True, "=", ("?e2", "?e1"))]))
    return out


def entails(premises, goal, prover=None):
    prover = prover or Prover()
    kb = []
    for f in premises:
        kb += to_clauses(f)
    neg_goal = to_clauses(("not", goal))
    kb += equality_axioms(kb + neg_goal)
    ok, _ = prover.refute(kb, neg_goal)
    return ok


## 2. Rule based reasoning (facts and rules)

Atoms are tuples like `('young', 'bob')` or `('chase', 'cat', 'bear')`. A rule is a list of body literals and a head literal. The first cell also has the pattern parser for simple English theories (RuleTaker sentences).

In [6]:
class Rule:
    def __init__(self, body, head, text=""):
        self.body = body
        self.head = head
        self.text = text

    def __repr__(self):
        b = " & ".join(lit_str(l) for l in self.body)
        return f"{b} -> {lit_str(self.head)}"


def atom_str(a):
    return f"{a[0]}({', '.join(a[1:])})"


def lit_str(lit):
    a, pos = lit
    return atom_str(a) if pos else "~" + atom_str(a)


def is_var(t):
    return isinstance(t, str) and t.startswith("?")


In [7]:
VAR_WORDS = {"someone", "something", "they", "it", "he", "she", "you", "everyone",
             "everything", "anyone", "anything", "people", "things", "person", "thing",
             "a person", "a thing", "somebody", "everybody", "anybody", "them", "one"}
PLURAL_NOUNS = {"things", "people", "persons", "ones", "animals", "creatures", "objects"}
COPULAS = {"is", "are", "am", "was", "were", "be"}


def lemma(verb):
    v = verb.lower()
    if v in ("is", "are", "am"):
        return "be"
    if v == "has":
        return "have"
    if v == "does":
        return "do"
    if len(v) > 4 and v.endswith("ies"):
        return v[:-3] + "y"
    if v.endswith(("ches", "shes", "sses", "xes", "zes")):
        return v[:-2]
    if v.endswith("s") and not v.endswith("ss") and len(v) > 2:
        return v[:-1]
    return v


def singular(noun):
    if noun.endswith("ies"):
        return noun[:-3] + "y"
    if noun.endswith(("ches", "shes", "sses", "xes")):
        return noun[:-2]
    if noun.endswith("s") and not noun.endswith("ss"):
        return noun[:-1]
    return noun


def plural_to_singular(word):
    if word.endswith(("ss", "us", "is")) or not word.endswith("s"):
        return word
    return singular(word)


def norm_entity(text):
    t = text.strip().lower()
    t = re.sub(r"^(the|a|an) ", "", t)
    return t.replace(" ", "_")


def parse_subject(text):
    t = text.strip().lower()
    if t in VAR_WORDS:
        return "?x"
    if re.fullmatch(r"the [a-z]+( [a-z]+){0,2}", t):
        return norm_entity(t)
    if re.fullmatch(r"[a-z]+", t) and t not in STOP_WORDS:
        return t
    return None


STOP_WORDS = {"not", "and", "or", "if", "then", "all", "the", "a", "an", "is", "are",
              "does", "do", "who", "that", "which", "no", "some", "every", "each"}


def parse_object(text):
    t = text.strip().lower()
    if not t or t.split()[0] in COPULAS or (t.split()[0] in STOP_WORDS and t.split()[0] not in ("the", "a", "an")):
        return None
    if t in VAR_WORDS:
        return "?x"
    if re.fullmatch(r"(the |a |an )[a-z]+( [a-z]+){0,2}|[a-z]+", t):
        return norm_entity(t)
    return None


def parse_full_clause(text):
    t = text.strip().lower()
    t = t.replace("doesn't", "does not").replace("don't", "do not")
    t = t.replace("isn't", "is not").replace("aren't", "are not")
    m = re.fullmatch(r"(.+?) (is|are|am) (not )?(a |an )?([a-z]+)", t)
    if m:
        subj = parse_subject(m.group(1))
        if subj is not None:
            return subj, [((m.group(5), subj), m.group(3) is None)]
    words = t.split()
    for i in range(1, len(words) - 1):
        subj = parse_subject(" ".join(words[:i]))
        if subj is None:
            continue
        rest = words[i:]
        positive = True
        if rest[:2] in (["does", "not"], ["do", "not"]):
            positive = False
            rest = rest[2:]
        if len(rest) < 2 or rest[0] in COPULAS or rest[0] in ("not", "and", "does", "do"):
            continue
        obj = parse_object(" ".join(rest[1:]))
        if obj is not None:
            return subj, [((lemma(rest[0]), subj, obj), positive)]
    return None


def parse_conjunction(text, subj=None):
    text = text.strip().lower().replace(", and ", " and ").replace(",", " and ")
    text = re.sub(r"\s+", " ", text)
    lits = []
    last_kind = "be"
    for piece in text.split(" and "):
        piece = piece.strip()
        if not piece:
            continue
        r = parse_full_clause(piece)
        if r:
            subj, ls = r
            lits += ls
            last_kind = "be" if len(ls[0][0]) == 2 else "rel"
            continue
        if subj is None:
            return None
        m = re.fullmatch(r"(not )?([a-z]+)", piece)
        if m and last_kind == "be":
            lits.append(((m.group(2), subj), m.group(1) is None))
            continue
        r = parse_full_clause("someone " + piece)
        if r:
            ls = [((a[0], subj) + a[2:], p) for (a, p) in r[1]]
            lits += ls
            last_kind = "be" if len(ls[0][0]) == 2 else "rel"
            continue
        return None
    return lits


def parse_adj_list(text):
    t = text.replace(",", " and ")
    out = []
    for p in t.split(" and "):
        p = p.strip()
        if not p:
            continue
        m = re.fullmatch(r"(not )?([a-z]+)", p)
        if not m:
            return None
        out.append((m.group(2), m.group(1) is None))
    return out


def parse_generic(low):
    m = re.fullmatch(r"(?:all |every |any )?([a-z ,]+?) are (not )?([a-z ,]+)", low)
    if not m:
        return None
    words = m.group(1).replace(",", " , ").split()
    if not words:
        return None
    if words[-1] in PLURAL_NOUNS:
        body_adj = parse_adj_list(" ".join(words[:-1]))
    elif words[-1].endswith("s"):
        body_adj = parse_adj_list(" ".join(words[:-1])) if len(words) > 1 else []
        if body_adj is not None:
            body_adj = body_adj + [(singular(words[-1]), True)]
    else:
        return None
    heads = parse_adj_list(m.group(3))
    if not body_adj or not heads:
        return None
    if words[-1] not in PLURAL_NOUNS:
        heads = [(plural_to_singular(h), p) for h, p in heads]
    neg = m.group(2) is not None
    body = [((a, "?x"), p) for a, p in body_adj]
    return [Rule(body, ((h, "?x"), p and not neg), low) for h, p in heads]


def parse_sentence(sent):
    s = sent.strip().rstrip(".!").strip()
    low = re.sub(r"\s+", " ", s.lower())

    m = re.fullmatch(r"if (.+?),? then (.+)", low)
    if m:
        body = parse_conjunction(m.group(1))
        if body is None:
            return None
        subj = "?x" if any("?x" in a for a, _ in body) else None
        head = parse_conjunction(m.group(2), subj)
        if head is None:
            return None
        return "rule", [Rule(body, h, s) for h in head]

    m = re.fullmatch(r"(?:all |every )?(someone|something|everyone|everything|people|things|anyone|anything|a person|a thing) (?:who|that|which) (.+) (?:is|are) (not )?([a-z ,]+)", low)
    if m:
        inner = m.group(2)
        if re.match(r"(is|are) ", inner):
            body = parse_conjunction("someone " + inner)
        else:
            body = parse_conjunction("someone " + inner)
        heads = parse_adj_list(m.group(4))
        if body and heads:
            neg = m.group(3) is not None
            return "rule", [Rule(body, ((h, "?x"), p and not neg), s) for h, p in heads]

    rules = parse_generic(low)
    if rules:
        return "rule", rules

    lits = parse_conjunction(low)
    if lits is not None and all(not is_var(t) for a, _ in lits for t in a[1:]):
        return "fact", lits
    return None


def split_sentences(text):
    parts = re.split(r"(?<=[.!?])\s+|\n+", text.strip())
    return [p for p in parts if p.strip()]


def parse_theory(context):
    facts, rules, failed = [], [], []
    for s in split_sentences(context):
        r = parse_sentence(s)
        if r is None:
            failed.append(s)
        elif r[0] == "fact":
            facts += r[1]
        else:
            rules += r[1]
    return facts, rules, failed


### Forward chaining

Closed world: `not X` holds if X cannot be derived (negation as failure). Rules are run stratum by stratum so a negated atom is only tested after everything that could derive it has finished. If the predicates have a cycle through negation, the rules are grounded and the well-founded model is computed with the alternating fixpoint.

In [8]:
def unify_atom(pattern, fact, theta):
    if len(pattern) != len(fact) or pattern[0] != fact[0]:
        return None
    theta = dict(theta)
    for p, f in zip(pattern[1:], fact[1:]):
        if is_var(p):
            if p in theta:
                if theta[p] != f:
                    return None
            else:
                theta[p] = f
        elif p != f:
            return None
    return theta


def subst(atom, theta):
    return tuple(theta.get(t, t) if is_var(t) else t for t in atom)


def stratify(rules, by_atom=False):
    key = (lambda a: a) if by_atom else (lambda a: a[0])
    preds = set()
    edges = []
    for r in rules:
        if not r.head[1]:
            continue
        h = key(r.head[0])
        preds.add(h)
        for a, pos in r.body:
            preds.add(key(a))
            edges.append((key(a), h, 0 if pos else 1))
    level = {p: 0 for p in preds}
    for _ in range(len(preds) + 2):
        changed = False
        for src, dst, w in edges:
            if level[dst] < level[src] + w:
                level[dst] = level[src] + w
                changed = True
                if level[dst] > len(preds):
                    return None
        if not changed:
            return level
    return None


class KB:
    def __init__(self, facts, rules):
        self.rules = rules
        self.pos = set()
        self.neg = set()
        self.how = {}
        for a, p in facts:
            if p:
                self.pos.add(a)
                self.how[a] = ("fact", [])
            else:
                self.neg.add(a)
                self.how[("~",) + a] = ("fact", [])
        self.domain = set()
        for a, _ in facts:
            self.domain.update(a[1:])
        for r in rules:
            for a, _ in r.body + [r.head]:
                self.domain.update(t for t in a[1:] if not is_var(t))
        self.index = defaultdict(set)
        for a in self.pos:
            self.index[a[0]].add(a)
        self.stratified = True
        self.rounds = 0

    def _match(self, body, theta, used):
        if not body:
            yield theta, used
            return
        (a, pos), rest = body[0], body[1:]
        if pos:
            for f in list(self.index.get(a[0], ())):
                t2 = unify_atom(a, f, theta)
                if t2 is not None:
                    yield from self._match(rest, t2, used + [f])
        else:
            free = [t for t in a[1:] if is_var(t) and t not in theta]
            if free:
                for c in self.domain:
                    t2 = dict(theta)
                    t2[free[0]] = c
                    yield from self._match(body, t2, used)
                return
            g = subst(a, theta)
            if g not in self.pos:
                yield from self._match(rest, theta, used + [("~",) + g])

    def _add(self, lit, rule, used):
        a, pos = lit
        if pos:
            if a in self.pos:
                return False
            self.pos.add(a)
            self.index[a[0]].add(a)
            self.how[a] = (rule, used)
            return True
        if a in self.neg:
            return False
        self.neg.add(a)
        self.how[("~",) + a] = (rule, used)
        return True

    def _run(self, rules):
        changed = True
        while changed:
            changed = False
            self.rounds += 1
            for r in rules:
                body = sorted(r.body, key=lambda l: not l[1])
                for theta, used in list(self._match(body, {}, [])):
                    head = subst(r.head[0], theta)
                    if any(is_var(t) for t in head[1:]):
                        continue
                    if self._add((head, r.head[1]), r, used):
                        changed = True

    def ground(self, rule):
        vs = sorted({t for a, _ in rule.body + [rule.head] for t in a[1:] if is_var(t)})
        for combo in itertools.product(sorted(self.domain), repeat=len(vs)):
            theta = dict(zip(vs, combo))
            body = [(subst(a, theta), p) for a, p in rule.body]
            yield Rule(body, (subst(rule.head[0], theta), rule.head[1]), rule.text)

    def _least_model(self, ground, assumed):
        model = set(self.pos)
        how = {}
        changed = True
        while changed:
            changed = False
            self.rounds += 1
            for g in ground:
                h = g.head[0]
                if h in model:
                    continue
                ok = True
                for a, p in g.body:
                    if (p and a not in model) or (not p and a in assumed):
                        ok = False
                        break
                if ok:
                    model.add(h)
                    how[h] = (g, [a if p else ("~",) + a for a, p in g.body])
                    changed = True
        return model, how

    def forward_chain_wfs(self):
        self.stratified = False
        ground = [g for r in self.rules if r.head[1] for g in self.ground(r)]
        over, _ = self._least_model(ground, set())
        while True:
            under, how = self._least_model(ground, over)
            new_over, _ = self._least_model(ground, under)
            if new_over == over:
                break
            over = new_over
        for a in under - self.pos:
            self._add((a, True), *how[a])

    def forward_chain(self):
        levels = stratify(self.rules)
        if levels is None:
            self.forward_chain_wfs()
            self._run([r for r in self.rules if not r.head[1]])
            return
        for lv in sorted(set(levels.values())):
            self._run([r for r in self.rules if r.head[1] and levels[r.head[0][0]] == lv])
        self._run([r for r in self.rules if not r.head[1]])

    def forward_chain_naive(self):
        self._run([r for r in self.rules if r.head[1]])
        self._run([r for r in self.rules if not r.head[1]])

    def ask_cwa(self, lit):
        a, pos = lit
        return (a in self.pos) == pos


### Backward chaining

Goal driven version, used for comparison and for printing proof trees.

In [9]:
class BackwardChainer:

    def __init__(self, facts, rules, max_depth=40):
        self.facts = defaultdict(list)
        for a, p in facts:
            if p:
                self.facts[a[0]].append(a)
        self.rules = defaultdict(list)
        for r in rules:
            if r.head[1]:
                self.rules[r.head[0][0]].append(r)
        self.domain = set()
        for a, _ in facts:
            self.domain.update(a[1:])
        for r in rules:
            for a, _ in r.body + [r.head]:
                self.domain.update(t for t in a[1:] if not is_var(t))
        self.proved = {}
        self.max_depth = max_depth
        self.calls = 0
        self.counter = itertools.count()

    def rename(self, rule):
        n = next(self.counter)
        m = {}
        def r(t):
            if is_var(t):
                m.setdefault(t, f"{t}_{n}")
                return m[t]
            return t
        head = (tuple([rule.head[0][0]] + [r(t) for t in rule.head[0][1:]]), rule.head[1])
        body = [(tuple([a[0]] + [r(t) for t in a[1:]]), p) for a, p in rule.body]
        return body, head

    def prove(self, atom, path=()):
        if atom in self.proved:
            return self.proved[atom]
        if atom in path or len(path) > self.max_depth:
            return None
        self.calls += 1
        for f in self.facts.get(atom[0], ()):
            if f == atom:
                self.proved[atom] = ("fact", atom, [])
                return self.proved[atom]
        for rule in self.rules.get(atom[0], ()):
            body, head = self.rename(rule)
            theta = unify_atom(head[0], atom, {})
            if theta is None:
                continue
            for th, subproofs in self.solve(body, theta, path + (atom,)):
                proof = ("rule", atom, subproofs, rule)
                self.proved[atom] = proof
                return proof
        return None

    def solve(self, goals, theta, path):
        if not goals:
            yield theta, []
            return
        goals = sorted(goals, key=lambda l: (not l[1], sum(is_var(theta.get(t, t)) for t in l[0][1:])))
        (a, pos), rest = goals[0], goals[1:]
        a = subst(a, theta)
        free = [t for t in a[1:] if is_var(t)]
        if free:
            for c in sorted(self.domain):
                yield from self.solve(goals, dict(theta, **{free[0]: c}), path)
            return
        if pos:
            p = self.prove(a, path)
            if p is not None:
                for th, sub in self.solve(rest, theta, path):
                    yield th, [p] + sub
        else:
            if self.prove(a, path) is None:
                for th, sub in self.solve(rest, theta, path):
                    yield th, [("naf", a, [])] + sub

    def ask(self, lit):
        a, pos = lit
        return (self.prove(a) is not None) == pos


def print_proof(proof, indent=0):
    kind, atom = proof[0], proof[1]
    pad = "    " * indent
    if kind == "fact":
        print(f"{pad}{atom_str(atom)}   [given fact]")
    elif kind == "naf":
        print(f"{pad}not {atom_str(atom)}   [cannot be proved, negation as failure]")
    else:
        print(f"{pad}{atom_str(atom)}   [rule: {proof[3].text or repr(proof[3])}]")
        for p in proof[2]:
            print_proof(p, indent + 1)


## 3. Counterfactual questions (CounterBench)

The causal model can be English (`A and B together cause not C`) or equations (`C = ¬A ∨ ¬B`). For *would Y occur if not X instead of X*: find root values that agree with what is known about the actual world (abduction, enumerating the roots), set the intervened variables (action), propagate forward through the equations (prediction).

In [10]:
def parse_side(text):
    t = text.strip()
    t = re.sub(r"\s+together$", "", t)
    op = "or" if re.search(r"\bor\b", t) else "and"
    parts = re.split(r",\s*|\s+and\s+|\s+or\s+", t)
    lits = []
    for p in parts:
        p = p.strip()
        if not p:
            continue
        neg = p.lower().startswith("not ")
        name = p[4:].strip() if neg else p
        lits.append((name.lower(), not neg))
    return op, lits


def split_clauses(text):
    t = text.strip()
    t = re.sub(r"^we know that\s+", "", t, flags=re.I)
    out = []
    for piece in re.split(r",\s*|\.\s+", t):
        piece = re.sub(r"^and\s+", "", piece.strip().rstrip("."))
        if not piece:
            continue
        if re.search(r"\bcauses?\b", piece) or not out or re.search(r"observed|~", piece, re.I):
            out.append(piece)
        else:
            out[-1] += " and " + piece
    return out


EQ_RE = re.compile(r"^\s*(\w+)\s*(?::=|=|↔|<->|<=>)\s*(.+)$")


def is_equation_model(text):
    lines = split_formulas(text)
    return bool(lines) and sum(1 for l in lines if EQ_RE.match(l)) >= max(1, len(lines) // 2)


def parse_equation_context(text):
    eqs, evidence = {}, {}
    for line in split_formulas(text):
        line = re.sub(r"^(we )?observed:?\s*", "", line.strip(), flags=re.I)
        m = EQ_RE.match(line)
        if m:
            eqs[m.group(1).lower()] = ("fol", parse_fol(m.group(2)))
            continue
        f = parse_fol(line)
        pos = True
        while f[0] == "not":
            f, pos = f[1], not pos
        if f[0] == "atom" and not f[2]:
            evidence[f[1]] = pos
    return eqs, evidence


def eval_formula(f, values):
    op = f[0]
    if op == "atom":
        return values[f[1]]
    if op == "not":
        return not eval_formula(f[1], values)
    a, b = eval_formula(f[1], values), eval_formula(f[2], values)
    return {"and": a and b, "or": a or b, "imp": (not a) or b, "iff": a == b, "xor": a != b}[op]


def formula_atoms(f):
    if f[0] == "atom":
        return [f[1]]
    return [x for g in f[1:] if isinstance(g, tuple) for x in formula_atoms(g)]


def parse_causal_context(text):
    if is_equation_model(text):
        return parse_equation_context(text)
    eqs = {}
    evidence = {}
    for c in split_clauses(text):
        m = re.match(r"we observed (.+)", c, re.I)
        if m:
            for name, val in parse_observation(m.group(1)):
                evidence[name] = val
            continue
        if "~" in c:
            continue
        m = re.fullmatch(r"(.+?)\s+causes?\s+(.+)", c)
        if not m:
            continue
        op, causes = parse_side(m.group(1))
        _, effects = parse_side(m.group(2))
        for eff, pos in effects:
            if pos:
                new = (op, causes)
            else:
                flip = "or" if op == "and" else "and"
                new = (flip, [(n, not p) for n, p in causes])
            if eff in eqs:
                old = eqs[eff]
                new = ("or2", [old, new])
            eqs[eff] = new
    return eqs, evidence


def parse_observation(text):
    out = []
    for p in re.split(r"\s+and\s+|,\s*", text.strip().rstrip(".")):
        p = re.sub(r"\s+(occurs?|occurred)$", "", p.strip())
        neg = False
        if p.lower().startswith("not "):
            neg, p = True, p[4:]
        m = re.fullmatch(r"(\w+)\s+not", p)
        if m:
            neg, p = True, m.group(1)
        out.append((p.strip().lower(), not neg))
    return out


def eval_eq(eq, values):
    op, items = eq
    if op == "fol":
        return eval_formula(items, values)
    if op == "or2":
        return any(eval_eq(e, values) for e in items)
    vals = [values[n] == p for n, p in items]
    return all(vals) if op == "and" else any(vals)


def topo_order(eqs, variables):
    order, seen = [], set()

    def visit(v, stack):
        if v in seen:
            return
        if v in stack:
            return
        stack.add(v)
        for p in parents(eqs.get(v)):
            visit(p, stack)
        stack.discard(v)
        seen.add(v)
        order.append(v)

    for v in sorted(variables):
        visit(v, set())
    return order


def parents(eq):
    if eq is None:
        return []
    op, items = eq
    if op == "fol":
        return formula_atoms(items)
    if op == "or2":
        return [p for e in items for p in parents(e)]
    return [n for n, _ in items]


def all_vars(eqs, extra=()):
    vs = set(eqs) | set(extra)
    for e in eqs.values():
        vs.update(parents(e))
    return vs


def propagate(eqs, order, roots, interventions):
    values = {v: False for v in order}
    for _ in range(len(order) + 1):
        changed = False
        for v in order:
            if v in interventions:
                new = interventions[v]
            elif v in eqs:
                new = eval_eq(eqs[v], values)
            else:
                new = roots[v]
            if new != values[v]:
                values[v] = new
                changed = True
        if not changed:
            break
    return values


def consistent_worlds(eqs, order, roots, evidence):
    for combo in itertools.product([True, False], repeat=len(roots)):
        r = dict(zip(roots, combo))
        vals = propagate(eqs, order, r, {})
        if all(vals[k] == v for k, v in evidence.items()):
            yield r


def parse_question(q):
    q = q.strip()
    m = re.fullmatch(r"Would (\w+) occur if not (\w+) instead of (\w+)\??", q, re.I)
    if m:
        return "instead", m.group(1).lower(), [(m.group(2).lower(), False)]
    m = re.fullmatch(r"Assume (.+?),? and based on this assumption, further suppose (.+?)\. Would (\w+) occur\??", q, re.I)
    if m:
        return "nested", m.group(3).lower(), parse_observation(m.group(1)) + parse_observation(m.group(2))
    m = re.fullmatch(r"Would (\w+) occur if (.+?)\??", q, re.I)
    if m:
        return "joint", m.group(1).lower(), parse_observation(m.group(2))
    return None


def is_causal_problem(context, question):
    if parse_question(question) is None:
        return False
    return re.search(r"\bcauses?\b", context) is not None or is_equation_model(context)


def solve_counterfactual(context, question, trace=False):
    eqs, evidence = parse_causal_context(context)
    kind, target, ints = parse_question(question)
    vs = all_vars(eqs, [target] + [n for n, _ in ints] + list(evidence))
    order = topo_order(eqs, vs)
    roots = [v for v in order if v not in eqs]
    known = dict(evidence)
    if kind == "instead":
        for n, val in ints:
            known[n] = not val
    worlds = list(consistent_worlds(eqs, order, roots, known))
    if not worlds:
        worlds = [dict.fromkeys(roots, True)]
    results = [propagate(eqs, order, w, dict(ints))[target] for w in worlds]
    answer = all(results)
    if trace:
        print("structural equations:")
        for v in order:
            if v in eqs:
                print("   ", v, "=", eq_str(eqs[v]))
        print("roots:", roots, " known facts:", known)
        w = worlds[0]
        print("actual world:        ", {k: int(v) for k, v in propagate(eqs, order, w, {}).items()})
        print("after do(" + ", ".join(f"{n}={int(v)}" for n, v in ints) + "):",
              {k: int(v) for k, v in propagate(eqs, order, w, dict(ints)).items()})
        print(f"{len(worlds)} consistent world(s), {target} = {answer}")
    return answer


def eq_str(eq):
    op, items = eq
    if op == "fol":
        return formula_text(items)
    if op == "or2":
        return " or ".join("(" + eq_str(e) + ")" for e in items)
    return f" {op} ".join(n if p else "not " + n for n, p in items)


def formula_text(f):
    op = f[0]
    if op == "atom":
        return f[1]
    if op == "not":
        return "not " + formula_text(f[1])
    return "(" + formula_text(f[1]) + f" {op} " + formula_text(f[2]) + ")"


## 4. English to logic (fallback when the input is not logic)

spaCy (small English model) gives lemmas, POS tags and the subject. Rules on top of that handle all / every / no / some, bare plurals, a/an generics, if-then, if and only if, either-or, neither-nor, relative clauses and pronouns. Predicate names are the sorted stemmed content words.

In [11]:
nlp = None

UNIV_DET = {"all", "every", "each", "any"}
NEG_DET = {"no"}
EXIST_DET = {"some", "several", "many", "most"}
VAR_SUBJ = {"someone", "somebody", "everyone", "everybody", "anyone", "anybody",
            "everything", "anything", "something", "people", "those", "whoever",
            "person", "one"}
NEG_SUBJ = {"nothing", "nobody", "none"}
PRONOUNS = {"he", "she", "it", "they", "him", "her", "them", "his", "its", "their", "this"}
SKIP_LEMMA = {"also", "very", "really", "just", "only", "both", "either", "neither",
              "then", "too", "so", "certain", "be", "do", "will", "would", "can", "could",
              "shall", "should", "may", "might", "must", "have"}
NEG_WORDS = {"not", "n't", "never", "no"}

_cache = {}
stemmer = None
LIGHT_WORDS = {"have", "locate", "also", "there", "kind", "type"}


def parse(text):
    global nlp, stemmer
    if nlp is None:
        import spacy
        from nltk.stem import PorterStemmer
        nlp = spacy.load("en_core_web_sm", disable=["ner"])
        stemmer = PorterStemmer()
    if text not in _cache:
        _cache[text] = nlp(text)
    return _cache[text]


def canon(tokens):
    words = []
    for i, t in enumerate(tokens):
        if t.is_punct or t.pos_ in ("DET", "PRON", "CCONJ", "SCONJ", "ADP", "PART", "AUX"):
            continue
        lem = t.lemma_.lower()
        if lem in SKIP_LEMMA and not (lem == "have" and t.pos_ == "VERB"):
            continue
        if lem in NEG_WORDS:
            continue
        lem = re.sub(r"[^a-z0-9]", "", lem)
        if lem in LIGHT_WORDS:
            continue
        if lem:
            words.append(stemmer.stem(lem))
    if not words:
        return None
    return "_".join(sorted(set(words)))


def atom(name, term):
    return ("atom", name, (term,))


def conj(fs):
    fs = [f for f in fs if f is not None]
    if not fs:
        return None
    out = fs[0]
    for f in fs[1:]:
        out = ("and", out, f)
    return out


def neg(f):
    return None if f is None else ("not", f)


def np_formula(tokens, term):
    toks = [t for t in tokens if not t.is_punct or t.text in ("-",)]
    while toks and (toks[0].pos_ == "DET" or toks[0].lower_ in ("also", "both", "either", "neither", "not")):
        toks = toks[1:]
    if not toks:
        return None
    if toks[0].lower_ in ("who", "that", "which"):
        return vp_formula(toks[1:], term)
    for i, t in enumerate(toks):
        if t.lower_ in ("who", "that", "which", "whose") and i > 0:
            head = np_formula(toks[:i], term)
            rel = vp_formula(toks[i + 1:], term)
            return conj([head, rel])
    first_adp = next((i for i, t in enumerate(toks) if t.pos_ == "ADP"), None)
    first_noun = next((i for i, t in enumerate(toks) if t.pos_ in ("NOUN", "PROPN", "NUM")), None)
    if first_adp is not None and (first_noun is None or first_adp < first_noun):
        name = canon(toks)
        return atom(name, term) if name else None
    seen_noun = False
    for i, t in enumerate(toks):
        if t.pos_ in ("NOUN", "PROPN", "NUM"):
            seen_noun = True
        elif seen_noun and (t.pos_ == "ADP" or t.tag_ in ("VBN", "VBG")):
            head = np_formula(toks[:i], term)
            mod = canon(toks[i:])
            return conj([head, atom(mod, term) if mod else None])
    nouns = [t for t in toks if t.pos_ in ("NOUN", "PROPN", "NUM")]
    adjs = [t for t in toks if t.pos_ in ("ADJ", "VERB") and t not in nouns]
    if not nouns:
        name = canon(toks)
        return atom(name, term) if name else None
    parts = []
    for a in adjs:
        name = canon([a])
        if name:
            parts.append(atom(name, term))
    full = canon(nouns)
    if full:
        parts.append(atom(full, term))
    last = canon([nouns[-1]])
    if len(nouns) > 1 and last and last != full:
        parts.append(atom(last, term))
    return conj(parts)


In [12]:
def split_top(tokens, word):
    idx = [i for i, t in enumerate(tokens) if t.lower_ == word]
    if not idx:
        return None
    parts, start = [], 0
    for i in idx:
        left, right = tokens[start:i], tokens[i + 1:]
        if not left or not right:
            continue
        if right[0].pos_ == "NUM" and left[-1].pos_ == "NUM":
            continue
        parts.append(left)
        start = i + 1
    parts.append(tokens[start:])
    if len(parts) < 2:
        return None
    out = []
    for p in parts:
        sub = []
        cur = []
        for t in p:
            if t.text == "," and word in ("and", "or"):
                if cur:
                    sub.append(cur)
                cur = []
            else:
                cur.append(t)
        if cur:
            sub.append(cur)
        out += sub
    return [p for p in out if p]


def vp_head_kind(tokens):
    for t in tokens:
        if t.lower_ in ("not", "n't", "never", "also", "both", "either", "neither"):
            continue
        if t.pos_ in ("AUX", "VERB"):
            return "verb"
        return "other"
    return "other"


def vp_formula(tokens, term, copula=False):
    toks = [t for t in tokens if not (t.is_punct and t.text not in (",",))]
    while toks and toks[-1].text == ",":
        toks = toks[:-1]
    if not toks:
        return None
    low = [t.lower_ for t in toks]

    if "neither" in low and "nor" in low:
        i, j = low.index("neither"), low.index("nor")
        pre = toks[:i]
        a = vp_formula(pre + toks[i + 1:j], term, copula)
        b = vp_formula(pre + toks[j + 1:], term, copula) if vp_head_kind(toks[j + 1:]) == "other" else vp_formula(toks[j + 1:], term, copula)
        return conj([neg(a), neg(b)])

    if "either" in low and "or" in low[low.index("either"):]:
        i = low.index("either")
        j = i + low[i:].index("or")
        pre = toks[:i]
        if pre and pre[-1].lower_ in ("not", "n't"):
            pre = pre[:-1]
            a = vp_formula(pre + toks[i + 1:j], term, copula)
            rest = toks[j + 1:]
            b = vp_formula(rest if vp_head_kind(rest) == "verb" else pre + rest, term, copula)
            return conj([neg(a), neg(b)]) if a is not None and b is not None else None
        a = vp_formula(pre + toks[i + 1:j], term, copula)
        rest = toks[j + 1:]
        b = vp_formula(rest if vp_head_kind(rest) == "verb" else pre + rest, term, copula)
        if a is not None and b is not None:
            return ("xor", a, b)
        return None

    for word, op in (("or", "or"), ("and", "and")):
        parts = split_top(toks, word)
        if parts:
            pre = []
            if vp_head_kind(parts[0]) == "verb":
                k = 0
                while k < len(parts[0]) and parts[0][k].pos_ in ("AUX",) or (k < len(parts[0]) and parts[0][k].lower_ in ("not", "n't")):
                    k += 1
                pre = parts[0][:k]
            if op == "or" and any(t.lower_ in ("not", "n't") for t in pre):
                pre = [t for t in pre if t.lower_ not in ("not", "n't")]
                parts = [[t for t in parts[0] if t.lower_ not in ("not", "n't")]] + parts[1:]
                fs = []
                for n, p in enumerate(parts):
                    if n > 0 and vp_head_kind(p) != "verb":
                        p = pre + p
                    fs.append(vp_formula(p, term, copula))
                if any(f is None for f in fs):
                    return None
                return conj([neg(f) for f in fs])
            fs = []
            for n, p in enumerate(parts):
                if n > 0 and vp_head_kind(p) != "verb":
                    p = pre + p
                fs.append(vp_formula(p, term, copula))
            if any(f is None for f in fs):
                return None
            out = fs[0]
            for f in fs[1:]:
                out = (op, out, f)
            return out

    negated = any(t.lower_ in ("not", "n't", "never") for t in toks)
    toks = [t for t in toks if t.lower_ not in ("not", "n't", "never")]
    is_cop = copula
    while toks and (toks[0].pos_ == "AUX" or toks[0].lower_ in ("also", "both", "always", "will", "can", "does", "do", "did")):
        if toks[0].lemma_ == "be":
            is_cop = True
        toks = toks[1:]
    if not toks:
        return None
    if is_cop:
        f = np_formula(toks, term)
    else:
        name = canon(toks)
        f = atom(name, term) if name else None
    if f is None:
        return None
    return neg(f) if negated else f


In [13]:
class Context:
    def __init__(self, indef="all"):
        self.last = None
        self.nvar = 0
        self.indef = indef

    def new_var(self):
        self.nvar += 1
        return f"?v{self.nvar}"


def subject_span(doc):
    root = [t for t in doc if t.dep_ == "ROOT"]
    if not root:
        return None
    root = root[0]
    subj = [c for c in root.children if c.dep_ in ("nsubj", "nsubjpass", "expl", "csubj")]
    if not subj:
        for t in doc:
            if t.tag_ in ("VBP", "VBZ", "VBD", "MD") and t.i > 0:
                return 0, t.i - 1
        for t in doc:
            if t.pos_ in ("VERB", "AUX") and t.i > 0:
                return 0, t.i - 1
        return None
    s = subj[0]
    left = min(t.i for t in s.subtree)
    right = max(t.i for t in s.subtree)
    return left, right


def const_name(tokens):
    words = [re.sub(r"[^a-z0-9]", "", t.lower_) for t in tokens if t.pos_ != "DET" and not t.is_punct]
    words = [w for w in words if w]
    return "_".join(words) if words else None


def analyse_subject(tokens, ctx):
    low = [t.lower_ for t in tokens]
    if not low:
        return None
    if low[0] in PRONOUNS and len(low) == 1:
        return "pron", None
    if low[0] in NEG_SUBJ or (low[0] == "no" and len(low) > 1 and low[1] == "one"):
        return "no", tokens[2:] if low[0] == "no" else tokens[1:]
    if low[0] in NEG_DET:
        return "no", tokens[1:]
    if low[0] in UNIV_DET:
        rest = tokens[1:]
        if rest and rest[0].lower_ in ("of", "the"):
            rest = rest[1:]
        if rest and rest[0].lower_ == "the":
            rest = rest[1:]
        return "all", rest
    if low[0] in EXIST_DET or low[:3] == ["at", "least", "one"]:
        return "some", tokens[1:] if low[0] in EXIST_DET else tokens[3:]
    if low[0] in ("a", "an"):
        return ctx.indef, tokens[1:]
    if low[0] in VAR_SUBJ:
        return "all", tokens[1:]
    if tokens[0].pos_ not in ("PROPN", "DET") and any(t.tag_ == "NNS" for t in tokens):
        return "all", tokens
    if "and" in low and all(t.pos_ in ("PROPN", "CCONJ", "PUNCT", "DET") for t in tokens):
        names, cur = [], []
        for t in tokens:
            if t.lower_ == "and" or t.text == ",":
                if cur:
                    names.append(const_name(cur))
                cur = []
            else:
                cur.append(t)
        if cur:
            names.append(const_name(cur))
        return "consts", [n for n in names if n]
    return "const", const_name(tokens)


def clause_formula(text, ctx, var_for_pron=None):
    doc = parse(text)
    span = subject_span(doc)
    if span is None:
        return None
    left, right = span
    subj = list(doc[left:right + 1])
    pred = [t for t in doc[right + 1:] if not (t.is_punct and t.text in (".", "!", "?"))]
    a = analyse_subject(subj, ctx)
    if a is None:
        return None
    kind, info = a
    if kind == "pron":
        term = var_for_pron or ctx.last
        if term is None:
            return None
        f = vp_formula(pred, term)
        return (f, "pron", term) if f else None
    if kind == "consts":
        fs = [vp_formula(pred, c) for c in info]
        if not fs or any(f is None for f in fs):
            return None
        ctx.last = info[-1]
        return conj(fs), "const", info[-1]
    if kind == "const":
        if info is None:
            return None
        ctx.last = info
        f = vp_formula(pred, info)
        return (f, "const", info) if f else None
    v = ctx.new_var()
    restr = np_formula(info, v) if info else None
    restr = drop_generic(restr)
    body = vp_formula(pred, v)
    if body is None:
        return None
    if kind == "all":
        return (("all", v, ("imp", restr, body)) if restr else ("all", v, body)), "all", v
    if kind == "no":
        return (("all", v, ("imp", restr, neg(body))) if restr else ("all", v, neg(body))), "no", v
    return ("ex", v, conj([restr, body])), "some", v


GENERIC = {"people", "person", "thing", "someone", "something", "one", "individual",
           "everyone", "anyone", "everything", "anything", "somebody", "everybody"}


GENERIC_STEMS = GENERIC | {"peopl", "someon", "someth", "individu", "everyon", "anyon",
                            "everyth", "anyth", "somebodi", "everybodi"}


def drop_generic(f):
    if f is None:
        return None
    if f[0] == "atom" and f[1] in GENERIC_STEMS:
        return None
    if f[0] == "and":
        a, b = drop_generic(f[1]), drop_generic(f[2])
        if a is None:
            return b
        if b is None:
            return a
        return ("and", a, b)
    return f


def strip_quant(f):
    if f[0] == "all" and f[2][0] == "imp":
        return f[1], f[2][1], f[2][2]
    if f[0] == "all":
        return f[1], None, f[2]
    return None


def replace_term(f, old, new):
    op = f[0]
    if op == "atom":
        return ("atom", f[1], tuple(new if a == old else a for a in f[2]))
    if op == "not":
        return ("not", replace_term(f[1], old, new))
    if op in ("all", "ex"):
        return (op, f[1], replace_term(f[2], old, new))
    return (op, replace_term(f[1], old, new), replace_term(f[2], old, new))


def conditional(cond, cons, ctx):
    saved = ctx.indef
    ctx.indef = "all"
    a = clause_formula(cond, ctx)
    ctx.indef = saved
    if a is None:
        return None
    fa, kind, v = a
    b = clause_formula(cons, ctx, var_for_pron=v)
    if b is None:
        return None
    fb, kind_b, vb = b
    if kind in ("all",):
        q = strip_quant(fa)
        if q is None:
            return None
        var, restr, body = q
        if kind_b == "all":
            qb = strip_quant(fb)
            if qb is None:
                return None
            fb = qb[2] if qb[1] is None else ("imp", qb[1], qb[2])
            fb = replace_term(fb, qb[0], var)
        return ("all", var, ("imp", conj([restr, body]), fb))
    return ("imp", fa, fb)


def clean(s):
    s = s.replace("’", "'").replace("“", '"').replace("”", '"')
    s = re.sub(r"(?<=\w)-(?=\w)", "", s)
    s = re.sub(r",?\s+or both\b", "", s)
    s = re.sub(r"\b(a|an|the)\s+(type|kind|sort|form)\s+of\s+", r"\1 ", s)
    s = re.sub(r"\s+", " ", s)
    return s


def sentence_formulas(sent, ctx):
    s = clean(sent.strip().rstrip(".!? ").strip())
    if not s:
        return []
    low = s.lower()
    if ";" in s:
        out = []
        for part in s.split(";"):
            out += sentence_formulas(part, ctx)
        return out
    m = re.match(r"^there(?: is|'s| are| exists?| exist)\s+(no |not any )?(?:a |an |some |at least one )?(.+)$", s, re.I)
    if m:
        v = ctx.new_var()
        body = drop_generic(np_formula(list(parse(m.group(2))), v))
        if body is None:
            return []
        f = ("ex", v, body)
        return [("not", f) if m.group(1) else f]
    m = re.match(r"^(.+?),?\s+if and only if\s+(.+)$", s, re.I)
    if m:
        f1 = conditional(m.group(2), m.group(1), ctx)
        f2 = conditional(m.group(1), m.group(2), ctx)
        return [f for f in (f1, f2) if f]
    m = re.match(r"^(?:if|when|whenever|once)\s+(.+?),\s*(?:then\s+)?(.+)$", s, re.I)
    if m:
        f = conditional(m.group(1), m.group(2), ctx)
        return [f] if f else []
    m = re.match(r"^(.+?)\s+(?:if|when|whenever)\s+(.+)$", s, re.I)
    if m and not low.startswith(("all ", "every ", "no ")):
        f = conditional(m.group(2), m.group(1), ctx)
        if f:
            return [f]
    m = re.match(r"^(.+?),?\s+(?:and|but)\s+((?:he|she|it|they)\s+.+)$", s, re.I)
    if m:
        return sentence_formulas(m.group(1), ctx) + sentence_formulas(m.group(2), ctx)
    r = clause_formula(s, ctx)
    return [r[0]] if r else []


def text_to_formulas(text, indef="all"):
    ctx = Context(indef)
    out, failed = [], []
    for sent in re.split(r"\n+|(?<=[.!?])\s+(?=[A-Z])", text.strip()):
        if not sent.strip():
            continue
        fs = sentence_formulas(sent, ctx)
        if fs:
            out += fs
        else:
            failed.append(sent)
    return out, failed


def conclusion_formula(text):
    fs, failed = text_to_formulas(text, indef="some")
    if failed or not fs:
        return None
    return conj(fs)


## 5. The agent

Order: FOL columns if the file has them -> causal question -> logic -> simple English theory (only if every sentence parses) -> English to FOL. For logic: Horn theory and literal query -> forward chaining (closed world); otherwise plain resolution. True only if the goal is proved and its negation is not. On the English route word subset axioms are added (if the words of predicate B are a strict subset of the words of A, add A(x) -> B(x)). Unknown / uncertain counts as False because the question is whether the statement is entailed.

In [14]:
_kb_cache = {}
_logic_cache = {}

LOGIC_HINT = re.compile(r"[∀∃¬∧∨→↔⊕]|:-|->|=>|<->|\b(forall|exists)\b|[A-Za-z_][\w']*\([^()]*\)")
STOP_WORDS_PRED = {"in", "of", "at", "to", "the", "a", "an", "is", "by", "for", "with", "on", "be", "are"}


def preds_of(f, out):
    if f[0] == "atom":
        out.add((f[1], len(f[2])))
    else:
        for x in f[1:]:
            if isinstance(x, tuple):
                preds_of(x, out)
    return out


def lexical_axioms(formulas):
    ps = set()
    for f in formulas:
        preds_of(f, ps)
    ps = [p for p, n in ps if n == 1 and p != "="]
    ws = {p: set(pred_words(p)) - STOP_WORDS_PRED for p in ps}
    out = []
    for a in ps:
        if not ws[a]:
            continue
        for b in ps:
            if b != a and ws[b] and ws[b] < ws[a]:
                out.append(("all", "?l", ("imp", ("atom", a, ("?l",)), ("atom", b, ("?l",)))))
    return out


def flatten_and(f):
    if f[0] == "and":
        return flatten_and(f[1]) + flatten_and(f[2])
    return [f]


def to_literal(f):
    pos = True
    while f[0] == "not":
        f = f[1]
        pos = not pos
    if f[0] != "atom" or f[1] == "=" or any(not isinstance(a, str) for a in f[2]):
        return None
    return ((f[1],) + tuple(f[2]), pos)


def strip_all(f):
    while f[0] == "all":
        f = f[2]
    return f


def to_horn(formulas):
    facts, rules = [], []
    for f in formulas:
        g = strip_all(f)
        if g[0] == "imp":
            body = [to_literal(x) for x in flatten_and(g[1])]
            head = [to_literal(x) for x in flatten_and(g[2])]
            if None in body or None in head:
                return None
            rules += [Rule(body, h, "") for h in head]
        else:
            lits = [to_literal(x) for x in flatten_and(g)]
            if None in lits or any(is_var(t) for a, _ in lits for t in a[1:]):
                return None
            facts += lits
    return facts, rules


def query_literals(goal):
    lits = [to_literal(x) for x in flatten_and(strip_all(goal))]
    if None in lits or any(is_var(t) for a, _ in lits for t in a[1:]):
        return None
    return lits


def parse_formulas(lines):
    out = []
    for line in lines:
        try:
            out.append(parse_fol(line))
        except (SyntaxError, IndexError, KeyError):
            parts = split_args(line, ",")
            if len(parts) > 1 and ":-" not in line:
                out += parse_formulas(parts)
    return out


def parse_logic(context, statement):
    if not LOGIC_HINT.search(str(context) + " " + str(statement)):
        return None
    if context not in _logic_cache:
        lines = split_formulas(context)
        prem = parse_formulas(lines)
        _logic_cache[context] = prem if len(prem) * 2 >= len(lines) else None
    prem = _logic_cache[context]
    if not prem:
        return None
    try:
        goal = parse_fol(re.sub(r"^\s*(query|q|statement|conclusion)\s*:\s*", "", statement, flags=re.I))
    except (SyntaxError, IndexError, KeyError):
        return None
    return prem, goal


def solve_logic(prem, goal, key=None):
    q = query_literals(goal)
    if q is not None:
        if key is not None and key in _kb_cache:
            kb = _kb_cache[key]
        else:
            horn = to_horn(prem)
            kb = None
            if horn is not None:
                kb = KB(*horn)
                kb.forward_chain()
            if key is not None:
                _kb_cache[key] = kb
        if kb is not None:
            return all(kb.ask_cwa(l) for l in q)
    if not entails(prem, goal, Prover()):
        return False
    return not entails(prem, ("not", goal), Prover())


TRIPLE_HINT = re.compile(r'\(\s*"[^"]*"\s+"[^"]*"\s+"[^"]*"\s+"[+-]"\s*\)')


def read_sexpr(tokens, i):
    out = []
    while i < len(tokens):
        t = tokens[i]
        if t == "(":
            sub, i = read_sexpr(tokens, i + 1)
            out.append(sub)
        elif t == ")":
            return out, i + 1
        else:
            out.append(t.strip('"') if t != "->" else t)
            i += 1
    return out, i


def triple_literal(t):
    subj, rel, obj, sign = [x.strip() for x in t]
    s = "?x" if subj.lower() in ("someone", "something", "it", "they") else norm_entity(subj)
    if rel.lower() == "is":
        return ((obj.lower().replace(" ", "_"), s), sign == "+")
    o = "?x" if obj.lower() in ("someone", "something", "it", "they") else norm_entity(obj)
    return ((lemma(rel), s, o), sign == "+")


def is_triple(x):
    return isinstance(x, list) and len(x) == 4 and all(isinstance(y, str) for y in x) and x[3] in ("+", "-")


def parse_triples(text):
    items, _ = read_sexpr(re.findall(r'"[^"]*"|\(|\)|->', text), 0)
    facts, rules = [], []

    def walk(xs):
        for x in xs:
            if is_triple(x):
                facts.append(triple_literal(x))
            elif isinstance(x, list) and "->" in x:
                k = x.index("->")
                body = [b for part in x[:k] for b in (part if not is_triple(part) else [part]) if is_triple(b)]
                head = [h for part in x[k + 1:] for h in (part if not is_triple(part) else [part]) if is_triple(h)]
                for h in head:
                    rules.append(Rule([triple_literal(b) for b in body], triple_literal(h), ""))
            elif isinstance(x, list):
                walk(x)

    walk(items)
    return facts, rules


def solve_triples(context, statement):
    key = ("triples", context)
    if key not in _kb_cache:
        kb = KB(*parse_triples(context))
        kb.forward_chain()
        _kb_cache[key] = kb
    q, _ = parse_triples(statement)
    if not q:
        return None
    return all(_kb_cache[key].ask_cwa(l) for l in q)


def solve_rule_based(context, statement):
    if context not in _kb_cache:
        facts, rules, failed = parse_theory(context)
        if failed:
            _kb_cache[context] = None
        else:
            kb = KB(facts, rules)
            kb.forward_chain()
            _kb_cache[context] = kb
    kb = _kb_cache[context]
    if kb is None:
        return None
    q = parse_sentence(statement)
    if q is None or q[0] != "fact":
        return None
    return all(kb.ask_cwa(lit) for lit in q[1])


def fol_problem(context, statement, premises_fol=None, conclusion_fol=None, lexical=True):
    if premises_fol and conclusion_fol:
        return parse_formulas(split_formulas(premises_fol)), parse_fol(conclusion_fol)
    prem, _ = text_to_formulas(context)
    goal = conclusion_formula(statement)
    if goal is None:
        return prem, None
    if lexical:
        prem = prem + lexical_axioms(prem + [goal])
    return prem, goal


def solve_english_fol(context, statement, lexical=True, check=True):
    prem, goal = fol_problem(context, statement, lexical=lexical)
    if goal is None:
        return False
    if not entails(prem, goal, Prover(max_clauses=3000, max_seconds=1.0)):
        return False
    if check:
        return not entails(prem, ("not", goal), Prover(max_clauses=3000, max_seconds=1.0))
    return True


def predict(context, statement, premises_fol=None, conclusion_fol=None):
    try:
        if premises_fol and conclusion_fol:
            prem, goal = fol_problem(context, statement, premises_fol, conclusion_fol)
            return solve_logic(prem, goal)
        if is_causal_problem(context, statement):
            return solve_counterfactual(context, statement)
        if TRIPLE_HINT.search(context):
            r = solve_triples(context, statement)
            if r is not None:
                return r
        logic = parse_logic(context, statement)
        if logic is not None:
            return solve_logic(logic[0], logic[1], key=("logic", context))
        r = solve_rule_based(context, statement)
        if r is not None:
            return r
        return solve_english_fol(context, statement)
    except Exception:
        return False


def clear_caches():
    _kb_cache.clear()
    _logic_cache.clear()
    _cache.clear()


examples = [
    ("Bob is young. Bob is kind. If someone is young, then they are smart. "
     "If someone is smart and kind, then they are successful.", "Bob is successful."),
    ("Young(bob). Kind(bob). ∀x (Young(x) → Smart(x)). ∀x (Smart(x) ∧ Kind(x) → Successful(x)).",
     "Successful(bob)"),
    ("young(bob)\nkind(bob)\nforall x (young(x) -> smart(x))\nforall x (smart(x) & kind(x) -> successful(x))",
     "~smart(bob)"),
    ("young(bob).\nkind(bob).\nsmart(X) :- young(X).\nsuccessful(X) :- smart(X), kind(X).", "successful(bob)"),
    ("∀x (Dog(x) → Mammal(x))\n∀x (Mammal(x) → Animal(x) ∨ Robot(x))\n¬Robot(rex)\nDog(rex)", "Animal(rex)"),
    ("Blaf = ¬Ziklo\nVork = Blaf ∧ Trune\nLumbo = Vork\nTrune", "Would Lumbo occur if not Ziklo instead of Ziklo?"),
    ("We know that Ziklo causes Blaf, Blaf causes Trune, and Trune causes Lumbo.",
     "Would Lumbo occur if not Ziklo instead of Ziklo?"),
    ("All dogs are mammals. All mammals are animals. Rex is a dog.", "Rex is an animal."),
]
for c, s in examples:
    print(predict(c, s), "|", s)


True | Bob is successful.
True | Successful(bob)
False | ~smart(bob)
True | successful(bob)
True | Animal(rex)
True | Would Lumbo occur if not Ziklo instead of Ziklo?
False | Would Lumbo occur if not Ziklo instead of Ziklo?
True | Rex is an animal.


## 6. Data

Files are read from `DATA_DIR` and downloaded from Hugging Face if missing. RuleTaker and CounterBench only come in English, so to test the logic input I convert them: RuleTaker theories through the parser of section 2, CounterBench models into equations. FOLIO comes with its own FOL annotations.

In [15]:
def data_path(name):
    os.makedirs(DATA_DIR, exist_ok=True)
    return os.path.join(DATA_DIR, name)


def get_ruletaker(split):
    path = data_path(f"tasksource__ruletaker__{split}.jsonl")
    if not os.path.exists(path):
        from datasets import load_dataset
        load_dataset("tasksource/ruletaker", split=split).to_json(path)
    return pd.read_json(path, lines=True)


def get_folio(split):
    path = data_path(f"tasksource__folio__{split}.jsonl")
    if not os.path.exists(path):
        from datasets import load_dataset
        load_dataset("tasksource/folio", split=split).to_json(path)
    return pd.read_json(path, lines=True)


def get_counterbench():
    rows = []
    for name in ["data_balanced_alpha_V1.json", "data_balanced_backdoor_V2.json"]:
        path = data_path("counterbench_" + name)
        if not os.path.exists(path):
            from huggingface_hub import hf_hub_download
            src = hf_hub_download("CounterBench/CounterBench", name, repo_type="dataset")
            with open(src, encoding="utf-8") as f, open(path, "w", encoding="utf-8") as g:
                g.write(f.read())
        with open(path, encoding="utf-8") as f:
            for i, e in enumerate(json.load(f)):
                e["file"] = name.split("_")[2]
                e["idx"] = i
                rows.append(e)
    return pd.DataFrame(rows)


if NEED_PUBLIC_DATA:
    rt_test = get_ruletaker("test")
    folio_val = get_folio("validation")
    cb = get_counterbench()
    print("RuleTaker test:", len(rt_test), " FOLIO validation:", len(folio_val), " CounterBench:", len(cb))
    print(rt_test.label.value_counts().to_dict())
    print(folio_val.label.value_counts().to_dict())
    print(cb.answer.value_counts().to_dict())
else:
    print("TEST_FILE is set, public data is not loaded")


RuleTaker test: 151911  FOLIO validation: 203  CounterBench: 1200
{'entailment': 76426, 'not entailment': 75485}
{'True': 72, 'Uncertain': 69, 'False': 62}
{'no': 605, 'yes': 595}


In [16]:
def pred_name(p):
    return "".join(w.capitalize() for w in p.split("_"))


def render_term(t, style):
    if is_var(t):
        return "X" if style == "prolog" else "x"
    return t


def render_lit(l, style):
    a, pos = l
    name = pred_name(a[0]) if style == "unicode" else a[0]
    s = f"{name}({', '.join(render_term(t, style) for t in a[1:])})"
    if pos:
        return s
    return {"unicode": "¬", "ascii": "~", "prolog": "\\+ "}[style] + s


def ruletaker_to_logic(context, style="unicode"):
    facts, rules, failed = parse_theory(context)
    out = [render_lit(f, style) for f in facts]
    for r in rules:
        if style == "prolog":
            out.append(f"{render_lit(r.head, style)} :- {', '.join(render_lit(b, style) for b in r.body)}")
            continue
        conj, arrow = (" ∧ ", " → ") if style == "unicode" else (" & ", " -> ")
        s = "(" + conj.join(render_lit(b, style) for b in r.body) + arrow + render_lit(r.head, style) + ")"
        if any(is_var(t) for a, _ in r.body + [r.head] for t in a[1:]):
            s = ("∀x " if style == "unicode" else "forall x ") + s
        out.append(s)
    if style == "prolog":
        return ".\n".join(out) + "."
    return "\n".join(out)


def ruletaker_query_to_logic(q, style="unicode"):
    return render_lit(parse_sentence(q)[1][0], style)


def causal_to_logic(given_info):
    eqs, ev = parse_causal_context(given_info)

    def side(eq):
        op, items = eq
        if op == "or2":
            return " ∨ ".join("(" + side(e) + ")" for e in items)
        sym = " ∧ " if op == "and" else " ∨ "
        return sym.join(("" if p else "¬") + n.capitalize() for n, p in items)

    lines = [f"{v.capitalize()} = {side(eq)}" for v, eq in eqs.items()]
    lines += [("" if val else "¬") + v.capitalize() for v, val in ev.items()]
    return "\n".join(lines)


if RUN_EXPERIMENTS:
    print(ruletaker_to_logic(rt_test.context.iloc[0])[:400])
    print("query:", rt_test.question.iloc[0], "->", ruletaker_query_to_logic(rt_test.question.iloc[0]))
    print()
    print(causal_to_logic(cb.given_info.iloc[300]))
    print(cb.question.iloc[300])


Big(anne)
Cold(anne)
Furry(anne)
Green(anne)
Red(anne)
Rough(anne)
Cold(erin)
Furry(erin)
Green(erin)
Round(erin)
∀x (Cold(x) → Furry(x))
∀x (Round(x) → Rough(x))
∀x (Green(x) ∧ Cold(x) → Furry(x))
∀x (Furry(x) → Round(x))
∀x (Red(x) → Big(x))
∀x (Rough(x) → Furry(x))
∀x (Rough(x) → Cold(x))
∀x (Red(x) → Round(x))
query: Anne is red. -> Red(anne)

Blaf = Ziklo
Trune = Ziklo
Vork = Blaf ∧ Trune
Lumbo = Vork
¬Trune
Would Lumbo occur if not Ziklo instead of Ziklo?


## 7. Experiments on the public data

### RuleTaker test split as logic (151,911 questions) in three notations

In [17]:
if RUN_EXPERIMENTS:
    logic_rows = []
    for style in ["unicode", "ascii", "prolog"]:
        clear_caches()
        rendered = {c: ruletaker_to_logic(c, style) for c in rt_test.context.unique()}
        queries = [ruletaker_query_to_logic(q, style) for q in rt_test.question]
        t0 = time.perf_counter()
        preds = [predict(rendered[c], q) for c, q in zip(rt_test.context, queries)]
        sec = time.perf_counter() - t0
        acc = 100 * (pd.Series(preds) == (rt_test.label == "entailment").values).mean()
        logic_rows.append({"notation": style, "example rule": rendered[rt_test.context.iloc[1]].split("\n")[-1],
                           "accuracy": round(acc, 3), "time s": round(sec, 1)})
    clear_caches()
    display(pd.DataFrame(logic_rows).set_index("notation"))


,example rule,accuracy,time s
notation,,,
unicode,∀x (Red(x) → Round(x)),100.0,7.8
ascii,forall x (red(x) -> round(x)),100.0,7.9
prolog,round(X) :- red(X).,100.0,8.1


### RuleTaker test split from English

`naive` runs all rules in one fixpoint, `forward` is the stratified version, `backward` is goal driven.

In [18]:
def run_ruletaker(df, method):
    correct = Counter()
    total = Counter()
    parse_time = 0.0
    reason_time = 0.0
    for ctx, g in df.groupby("context", sort=False):
        t0 = time.perf_counter()
        facts, rules, failed = parse_theory(ctx)
        queries = [parse_sentence(q)[1][0] for q in g.question]
        t1 = time.perf_counter()
        if method == "backward":
            kb = BackwardChainer(facts, rules)
            preds = [kb.ask(lit) for lit in queries]
        else:
            kb = KB(facts, rules)
            if method == "forward":
                kb.forward_chain()
            else:
                kb.forward_chain_naive()
            preds = [kb.ask_cwa(lit) for lit in queries]
        t2 = time.perf_counter()
        parse_time += t1 - t0
        reason_time += t2 - t1
        for p, lab, cfg in zip(preds, g.label, g.config):
            total[cfg] += 1
            correct[cfg] += p == (lab == "entailment")
    return correct, total, parse_time, reason_time


if RUN_EXPERIMENTS:
    rows = []
    for method in ["naive", "forward", "backward"]:
        correct, total, pt, rt = run_ruletaker(rt_test, method)
        row = {"method": method}
        for cfg in sorted(total):
            row[cfg] = round(100 * correct[cfg] / total[cfg], 2)
        row["all"] = round(100 * sum(correct.values()) / sum(total.values()), 3)
        row["errors"] = sum(total.values()) - sum(correct.values())
        row["parse s"] = round(pt, 2)
        row["reason s"] = round(rt, 2)
        rows.append(row)
    pd.set_option("display.width", 200)
    display(pd.DataFrame(rows).set_index("method").T)


method,naive,forward,backward
NatLang,100.00,100.00,100.00
depth-0,99.99,100.00,100.00
depth-1,100.00,100.00,100.00
depth-2,99.98,100.00,100.00
depth-3,99.91,100.00,100.00
depth-3ext,99.92,100.00,100.00
depth-3ext-NatLang,99.95,100.00,100.00
depth-5,99.89,100.00,100.00
all,99.95,100.00,100.00
errors,76.00,0.00,0.00


### FOLIO validation with its FOL annotations

Binary accuracy (True vs not True); 3-way is True / False / Uncertain.

In [19]:
if RUN_EXPERIMENTS:
    gold_problems = []
    for pf, cf, lab in zip(folio_val["premises-FOL"], folio_val["conclusion-FOL"], folio_val.label):
        try:
            gold_problems.append((parse_formulas(split_formulas(pf)), parse_fol(cf), lab))
        except SyntaxError:
            gold_problems.append((None, None, lab))


    def run_gold(mode):
        correct = correct3 = 0
        t0 = time.perf_counter()
        for prem, goal, lab in gold_problems:
            yes = no = False
            if prem is not None:
                try:
                    if mode == "final":
                        yes = solve_logic(prem, goal)
                    else:
                        extra = lexical_axioms(prem + [goal]) if mode == "axioms" else []
                        yes = entails(prem + extra, goal, Prover())
                        no = entails(prem + extra, ("not", goal), Prover())
                        yes = yes and not no
                except (OverflowError, RecursionError):
                    pass
            three = "True" if yes else "False" if no else "Uncertain"
            correct += yes == (lab == "True")
            correct3 += three == lab
        return 100 * correct / len(gold_problems), 100 * correct3 / len(gold_problems), time.perf_counter() - t0


    gold_rows = [{"setting": "always False (baseline)", "binary acc": round(100 * (folio_val.label != "True").mean(), 2)}]
    for name, mode in [("resolution only", "plain"), ("resolution + word subset axioms", "axioms"),
                       ("final: Horn -> forward chaining, else resolution", "final")]:
        acc, acc3, sec = run_gold(mode)
        row = {"setting": name, "binary acc": round(acc, 2), "time s": round(sec, 1)}
        if mode != "final":
            row["3-way acc"] = round(acc3, 2)
        gold_rows.append(row)
    display(pd.DataFrame(gold_rows).set_index("setting"))


,binary acc,time s,3-way acc
setting,,,
always False (baseline),64.53,NaN,NaN
resolution only,84.73,8.3,73.89
resolution + word subset axioms,83.74,8.8,72.91
"final: Horn -> forward chaining, else resolution",84.73,4.3,NaN


### Closed world or open world for Horn theories?

RuleTaker labels are closed world, FOLIO labels are open world, and both can arrive as Horn theories. Same Horn problems answered both ways:

In [20]:
if RUN_EXPERIMENTS:
    rows = []
    for name, problems in [("FOLIO validation (gold FOL)", gold_problems)]:
        n = cwa_ok = owa_ok = 0
        for prem, goal, lab in problems:
            if prem is None or to_horn(prem) is None or query_literals(goal) is None:
                continue
            kb = KB(*to_horn(prem))
            kb.forward_chain()
            cwa = all(kb.ask_cwa(l) for l in query_literals(goal))
            owa = entails(prem, goal, Prover()) and not entails(prem, ("not", goal), Prover())
            n += 1
            cwa_ok += cwa == (lab == "True")
            owa_ok += owa == (lab == "True")
        rows.append({"data": name, "Horn problems": n, "closed world correct": cwa_ok, "open world correct": owa_ok})

    sample = rt_test.groupby("config").sample(n=300, random_state=1)
    n = cwa_ok = owa_ok = 0
    for c, q, lab in zip(sample.context, sample.question, sample.label):
        prem = [parse_fol(f) for f in split_formulas(ruletaker_to_logic(c))]
        goal = parse_fol(ruletaker_query_to_logic(q))
        kb = KB(*to_horn(prem))
        kb.forward_chain()
        cwa = all(kb.ask_cwa(l) for l in query_literals(goal))
        owa = entails(prem, goal, Prover()) and not entails(prem, ("not", goal), Prover())
        n += 1
        cwa_ok += cwa == (lab == "entailment")
        owa_ok += owa == (lab == "entailment")
    rows.append({"data": "RuleTaker test sample (logic)", "Horn problems": n, "closed world correct": cwa_ok, "open world correct": owa_ok})
    display(pd.DataFrame(rows).set_index("data"))


,Horn problems,closed world correct,open world correct
data,,,
FOLIO validation (gold FOL),20,20,20
RuleTaker test sample (logic),2400,2400,1787


### CounterBench

In [21]:
if RUN_EXPERIMENTS:
    t0 = time.perf_counter()
    cb["pred"] = [solve_counterfactual(c, q) for c, q in zip(cb.given_info, cb.question)]
    cb_time = time.perf_counter() - t0
    cb["ok"] = cb.pred == (cb.answer == "yes")
    print(f"CounterBench accuracy: {100 * cb.ok.mean():.2f}%  ({cb.ok.sum()}/{len(cb)})  time {cb_time:.2f} s")
    display(cb.groupby("type").ok.agg(["mean", "sum", "count"]).assign(mean=lambda d: (100 * d["mean"]).round(1)))


CounterBench accuracy: 95.00%  (1140/1200)  time 0.11 s


,mean,sum,count
type,,,
backdoor,90.0,180,200
basic,100.0,250,250
conditional,92.0,230,250
joint,96.0,240,250
nested,96.0,240,250


In [22]:
if RUN_EXPERIMENTS:
    t0 = time.perf_counter()
    cb["pred_logic"] = [predict(causal_to_logic(c), q) for c, q in zip(cb.given_info, cb.question)]
    sec = time.perf_counter() - t0
    gold = cb.answer == "yes"
    print(f"CounterBench as equations: {100 * (cb.pred_logic == gold).mean():.2f}%  time {sec:.2f} s")
    print("same answer as the English version:", (cb.pred_logic == cb.pred).sum(), "of", len(cb))


CounterBench as equations: 95.00%  time 0.29 s
same answer as the English version: 1200 of 1200


### FOLIO validation from English (fallback route)

In [23]:
def run_english_folio(df, lexical=True, check=True):
    preds = []
    t0 = time.perf_counter()
    for premises, conclusion in zip(df.premises, df.conclusion):
        try:
            preds.append(solve_english_fol(premises, conclusion, lexical, check))
        except (SyntaxError, OverflowError, RecursionError):
            preds.append(False)
    sec = time.perf_counter() - t0
    acc = 100 * (pd.Series(preds, index=df.index) == (df.label == "True")).mean()
    return acc, sec


if RUN_EXPERIMENTS:
    eng_rows = [{"setting": "always False (baseline)", "binary acc": round(100 * (folio_val.label != "True").mean(), 2)}]
    for name, kw in [("English parser + resolution", dict(lexical=False, check=False)),
                     ("+ word subset axioms", dict(lexical=True, check=False)),
                     ("+ contradiction check", dict(lexical=True, check=True))]:
        acc, sec = run_english_folio(folio_val, **kw)
        eng_rows.append({"setting": name, "binary acc": round(acc, 2), "time s": round(sec, 1)})
    display(pd.DataFrame(eng_rows).set_index("setting"))


,binary acc,time s
setting,,
always False (baseline),64.53,NaN
English parser + resolution,67.49,17.1
+ word subset axioms,67.49,18.2
+ contradiction check,67.49,18.4


## 8. Reasoning traces

In [24]:
def show_theory(context):
    facts, rules, failed = parse_theory(context)
    print("facts:", ", ".join(lit_str(f) for f in facts))
    print("rules:")
    for r in rules:
        print("   ", r)


def trace_ruletaker(context, statement):
    show_theory(context)
    facts, rules, _ = parse_theory(context)
    lit = parse_sentence(statement)[1][0]
    bc = BackwardChainer(facts, rules)
    proof = bc.prove(lit[0])
    print("query:", statement, "->", lit_str(lit))
    if proof is None:
        print(atom_str(lit[0]), "cannot be proved, so under the closed world assumption it is false")
    else:
        print_proof(proof)
    print("answer:", bc.ask(lit))


if RUN_EXPERIMENTS:
    print("Example from the assignment sheet")
    trace_ruletaker(examples[0][0], examples[0][1])

    print()
    print("Same example given directly as logic")
    logic_kb = "Young(bob)\nKind(bob)\n∀x (Young(x) → Smart(x))\n∀x (Smart(x) ∧ Kind(x) → Successful(x))"
    print(logic_kb)
    facts, rules = to_horn([parse_fol(f) for f in split_formulas(logic_kb)])
    print_proof(BackwardChainer(facts, rules).prove(("successful", "bob")))

    print()
    print("RuleTaker depth-3 example with negation as failure")
    ex = rt_test[(rt_test.config == "depth-3") & (rt_test.label == "entailment")]
    ex = ex[ex.context.str.contains(" not ") & ex.question.str.contains("not")].iloc[0]
    trace_ruletaker(ex.context, ex.question)


Example from the assignment sheet
facts: young(bob), kind(bob)
rules:
    young(?x) -> smart(?x)
    smart(?x) & kind(?x) -> successful(?x)
query: Bob is successful. -> successful(bob)
successful(bob)   [rule: If someone is smart and kind, then they are successful]
    smart(bob)   [rule: If someone is young, then they are smart]
        young(bob)   [given fact]
    kind(bob)   [given fact]
answer: True

Same example given directly as logic
Young(bob)
Kind(bob)
∀x (Young(x) → Smart(x))
∀x (Smart(x) ∧ Kind(x) → Successful(x))
successful(bob)   [rule: smart(?x) & kind(?x) -> successful(?x)]
    smart(bob)   [rule: young(?x) -> smart(?x)]
        young(bob)   [given fact]
    kind(bob)   [given fact]

RuleTaker depth-3 example with negation as failure
facts: young(bob), quiet(dave), smart(dave), ~white(dave), white(erin), ~red(fiona), smart(fiona)
rules:
    red(?x) -> ~nice(?x)
    nice(dave) -> smart(dave)
    kind(fiona) & ~nice(fiona) -> young(fiona)
    young(?x) & quiet(?x) -> red(

In [25]:
if RUN_EXPERIMENTS:
    row = cb[cb.type == "conditional"].iloc[0]
    print(row.given_info)
    print(row.question, " gold:", row.answer)
    solve_counterfactual(row.given_info, row.question, trace=True)


We know that Ziklo causes not Blaf, Blaf and Trune together cause Vork, Vork causes Lumbo. We observed Trune
Would Lumbo occur if not Ziklo instead of Ziklo?  gold: yes
structural equations:
    blaf = not ziklo
    vork = blaf and trune
    lumbo = vork
roots: ['ziklo', 'trune']  known facts: {'trune': True, 'ziklo': True}
actual world:         {'ziklo': 1, 'blaf': 0, 'trune': 1, 'vork': 0, 'lumbo': 0}
after do(ziklo=0): {'ziklo': 0, 'blaf': 1, 'trune': 1, 'vork': 1, 'lumbo': 1}
1 consistent world(s), lumbo = True


In [26]:
def term_str(t):
    if isinstance(t, tuple):
        return t[0] + "(" + ", ".join(term_str(a) for a in t[1:]) + ")"
    return t


def clause_str(c):
    if not c:
        return "[]"
    lits = sorted(c, key=repr)
    return " | ".join(("" if s else "~") + p + "(" + ", ".join(term_str(a) for a in args) + ")" for s, p, args in lits)


def formula_str(f):
    op = f[0]
    if op == "atom":
        return f"{f[1]}({', '.join(term_str(a) for a in f[2])})"
    if op == "not":
        return "¬" + formula_str(f[1])
    if op in ("all", "ex"):
        return ("∀" if op == "all" else "∃") + f[1][1:] + " " + formula_str(f[2])
    sym = {"and": "∧", "or": "∨", "imp": "→", "xor": "⊕", "iff": "↔"}[op]
    return "(" + formula_str(f[1]) + f" {sym} " + formula_str(f[2]) + ")"


def show_refutation(parents):
    steps = []
    seen = set()

    def visit(c):
        k = clause_key(c)
        if k in seen:
            return
        seen.add(k)
        src = parents.get(k)
        if src is None or src == ("goal",):
            steps.append((c, "negated goal" if src else "premise"))
            return
        visit(src[0])
        visit(src[1])
        steps.append((c, "resolvent"))

    a, b = parents["empty"]
    visit(a)
    visit(b)
    for i, (c, how) in enumerate(steps, 1):
        print(f"{i:3d}. {clause_str(c)}   ({how})")
    print(f"{len(steps) + 1:3d}. []   (empty clause)")


def trace_logic(premises_text, goal_text):
    prem = [parse_fol(l) for l in split_formulas(premises_text)]
    goal = parse_fol(goal_text)
    for line, f in zip(split_formulas(premises_text), prem):
        print("   ", line)
    print("goal:", goal_text)
    if to_horn(prem) is not None and query_literals(goal) is not None:
        print("Horn theory -> forward chaining under the closed world assumption:", solve_logic(prem, goal))
        return
    kb = []
    for f in prem:
        kb += to_clauses(f)
    prover = Prover()
    ok, parents = prover.refute(kb, to_clauses(("not", goal)))
    print("not Horn -> resolution refutation. proved:", ok, " given clauses picked:", prover.steps)
    if ok:
        show_refutation(parents)


def trace_folio(premises, conclusion):
    prem, goal = fol_problem(premises, conclusion, lexical=False)
    print("premises:")
    for line, f in zip(premises.split("\n"), prem):
        print("   ", line.strip())
        print("       ", formula_str(f))
    print("conclusion:", conclusion)
    print("       ", formula_str(goal))
    kb = []
    for f in prem:
        kb += to_clauses(f)
    neg_goal = to_clauses(("not", goal))
    prover = Prover()
    ok, parents = prover.refute(kb, neg_goal)
    print("proved:", ok, " given clauses picked:", prover.steps)
    if ok:
        show_refutation(parents)


if RUN_EXPERIMENTS:
    pick = folio_val[(folio_val.label == "True") & (folio_val["premises-FOL"].str.count("\n") <= 5)]
    for pf, cf in zip(pick["premises-FOL"], pick["conclusion-FOL"]):
        prem = parse_formulas(split_formulas(pf))
        if to_horn(prem) is None and "∃" not in pf and entails(prem, parse_fol(cf)):
            print("FOLIO validation problem given as logic:")
            trace_logic(pf, cf)
            break

    print()
    print("The same idea from English (fallback route):")
    trace_folio("All people who regularly drink coffee are dependent on caffeine.\n"
                "People regularly drink coffee, or they don't want to be addicted to caffeine.\n"
                "No one who doesn't want to be addicted to caffeine is unaware that caffeine is a drug.\n"
                "Rina is unaware that caffeine is a drug.",
                "Rina is dependent on caffeine.")


FOLIO validation problem given as logic:
    ∀x (WildTurkey(x) → (EasternWildTurkey(x) ∨ OsceolaWildTurkey(x) ∨ GouldsWildTurkey(x) ∨ MerriamsWildTurkey(x) ∨ RiograndeWildTurkey(x) ∨ OcellatedWildTurkey(x)))
    ¬(EasternWildTurkey(tom))
    ¬(OsceolaWildTurkey(tom))
    ¬(GouldsWildTurkey(tom))
    ¬(MerriamsWildTurkey(tom) ∨ RiograndeWildTurkey(tom))
    WildTurkey(tom)
goal: OcellatedWildTurkey(tom)
not Horn -> resolution refutation. proved: True  given clauses picked: 7
  1. ~ocellatedwildturkey(tom)   (negated goal)
  2. ~wildturkey(?v132214) | easternwildturkey(?v132214) | gouldswildturkey(?v132214) | merriamswildturkey(?v132214) | ocellatedwildturkey(?v132214) | osceolawildturkey(?v132214) | riograndewildturkey(?v132214)   (premise)
  3. ~wildturkey(tom) | easternwildturkey(tom) | gouldswildturkey(tom) | merriamswildturkey(tom) | osceolawildturkey(tom) | riograndewildturkey(tom)   (resolvent)
  4. ~easternwildturkey(tom)   (premise)
  5. ~wildturkey(tom) | gouldswildturkey(tom) 

## 9. Final run and submission file

Without `TEST_FILE` this runs on a public evaluation file in logic form: 1000 RuleTaker test questions per depth config, all 203 FOLIO validation problems (gold FOL) and all 1200 CounterBench questions (equations). Caches are cleared first.

In [27]:
def to_bool(x):
    return str(x).strip().lower() in ("true", "entailment", "yes", "1", "t")


def as_text(x):
    if isinstance(x, (list, tuple)):
        return "\n".join(str(i) for i in x)
    s = str(x)
    if s.startswith("[") and s.endswith("]"):
        try:
            v = json.loads(s)
            if isinstance(v, list):
                return "\n".join(str(i) for i in v)
        except ValueError:
            pass
    return s


def load_problems(path):
    if path.endswith(".csv"):
        df = pd.read_csv(path)
    elif path.endswith(".jsonl"):
        df = pd.read_json(path, lines=True)
    else:
        df = pd.read_json(path)
    cols = {c.lower(): c for c in df.columns}

    def pick(names):
        for n in names:
            if n in cols:
                return cols[n]
        return None

    ctx = pick(["context", "premises", "theory", "given_info", "facts"])
    st = pick(["statement", "question", "query", "conclusion", "hypothesis"])
    lab = pick(["flag", "label", "answer", "gold"])
    idc = pick(["id", "example_id", "question_id", "qid"])
    out = pd.DataFrame({
        "id": df[idc] if idc else range(len(df)),
        "context": df[ctx].map(as_text),
        "statement": df[st].map(as_text),
    })
    pf, cf = pick(["premises-fol"]), pick(["conclusion-fol"])
    out["premises_fol"] = df[pf] if pf else None
    out["conclusion_fol"] = df[cf] if cf else None
    if lab:
        out["flag"] = df[lab].map(to_bool)
    if "source" in cols:
        out["source"] = df[cols["source"]]
    return out


def build_public_eval():
    rows = []
    sample = rt_test.groupby("config").sample(n=1000, random_state=42)
    for i, r in enumerate(sample.itertuples()):
        rows.append(("rt_%05d" % i, "ruletaker", ruletaker_to_logic(r.context),
                     ruletaker_query_to_logic(r.question), r.label == "entailment"))
    folio_rows = zip(folio_val["premises-FOL"], folio_val["conclusion-FOL"], folio_val.label)
    for i, (pf, cf, lab) in enumerate(folio_rows):
        rows.append(("folio_%03d" % i, "folio", pf, cf, lab == "True"))
    for r in cb.itertuples():
        rows.append((f"cb_{r.file}_{r.idx:04d}", "counterbench", causal_to_logic(r.given_info), r.question, r.answer == "yes"))
    df = pd.DataFrame(rows, columns=["id", "source", "context", "statement", "flag"])
    df.to_csv(data_path("public_eval_logic.csv"), index=False)
    return data_path("public_eval_logic.csv")


path = TEST_FILE if TEST_FILE else build_public_eval()
problems = load_problems(path)
clear_caches()

start = time.perf_counter()
preds = []
for c, s, pf, cf in zip(problems.context, problems.statement, problems.premises_fol, problems.conclusion_fol):
    pf = pf if isinstance(pf, str) else None
    cf = cf if isinstance(cf, str) else None
    preds.append(bool(predict(c, s, pf, cf)))
elapsed = time.perf_counter() - start
problems["pred"] = preds

print("test file:", os.path.basename(path))
print("number of queries:", len(problems))
print(f"total inference time: {elapsed:.2f} s ({1000 * elapsed / len(problems):.2f} ms per query)")
if "flag" in problems:
    acc = 100 * (problems.pred == problems.flag).mean()
    print(f"final accuracy: {acc:.2f}%")
    if "source" in problems:
        print((problems.pred == problems.flag).groupby(problems.source).mean().mul(100).round(2).to_string())

problems[["id"]].assign(flag=problems.pred).to_csv("submission.csv", index=False)
print("wrote submission.csv")


test file: public_eval_logic.csv
number of queries: 9403
total inference time: 7.66 s (0.81 ms per query)
final accuracy: 99.03%
source
counterbench     95.00
folio            84.73
ruletaker       100.00
wrote submission.csv
